In [0]:
import sys
import importlib

SRC_PATH = "/Workspace/Users/ariamostajeran99@gmail.com/stock_project_V2/stock-mlops-databricks/src"
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

import reporting_monitoring
importlib.reload(reporting_monitoring)

from reporting_monitoring import ReportingMonitoring

In [0]:
eval_by_stock_df = spark.table("p2_eval_by_stock").toPandas()
eval_confidence_by_stock_df = spark.table("p2_eval_confidence_by_stock").toPandas()
signal_summary_df = spark.table("p2_signal_summary").toPandas()
latest_signals_df = spark.table("p2_latest_signals").toPandas()
portfolio_summary_df = spark.table("p2_portfolio_backtest_summary").toPandas()

In [0]:
reporter = ReportingMonitoring(
    min_accuracy_threshold=0.52,
    min_lift_threshold=0.0,
    min_signal_count=10
)

model_health_df = reporter.model_health_report(
    eval_by_stock_df=eval_by_stock_df,
    eval_confidence_by_stock_df=eval_confidence_by_stock_df
)

latest_signal_report_df = reporter.latest_signal_report(latest_signals_df)

alerts_df = reporter.alert_flags(
    eval_by_stock_df=eval_by_stock_df,
    signal_summary_df=signal_summary_df,
    portfolio_summary_df=portfolio_summary_df
)

dashboard_summary_df = reporter.dashboard_summary(
    eval_by_stock_df=eval_by_stock_df,
    portfolio_summary_df=portfolio_summary_df,
    signal_summary_df=signal_summary_df
)

model_health_df


In [0]:
latest_signal_report_df

In [0]:
alerts_df

In [0]:
dashboard_summary_df

In [0]:
spark.createDataFrame(model_health_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_model_health")

spark.createDataFrame(latest_signal_report_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_latest_signal_report")

spark.createDataFrame(alerts_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_alerts")

spark.createDataFrame(dashboard_summary_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_dashboard_summary")